# Diagnostic Distractor SLM — Qwen3-1.7B (QLoRA)

Fine-tune Qwen3-1.7B to generate 3 misconception-tagged, numerically-consistent distractors for middle-school "Number" MCQs.

**How to run:** `Runtime → Change runtime type → T4 GPU`, then `Runtime → Run all`.

Flow: install → clone repo → load base model → **litmus** (show the base model fails) → **QLoRA fine-tune** → **base-vs-tuned eval**.

In [ ]:
%%capture
# Unsloth pulls a compatible torch/transformers/peft/trl/bitsandbytes/datasets stack.
!pip install -q unsloth
!pip install -q openai python-dotenv
# If this errors on Colab, use the official install snippet at https://unsloth.ai/docs

In [ ]:
import os, sys

if not os.path.exists("diagnostic-distractor-slm"):
    !git clone https://github.com/jsonjj/diagnostic-distractor-slm.git
%cd diagnostic-distractor-slm
sys.path.insert(0, ".")

MODEL_NAME  = "unsloth/Qwen3-1.7B-bnb-4bit"
MAX_SEQ_LEN = 2048
EPOCHS      = 3

# Optional: TrueFoundry key, only needed if you later run the judge (`--judge`).
# from getpass import getpass; os.environ["TFY_API_KEY"] = getpass("TFY_API_KEY: ")

In [ ]:
from unsloth import FastLanguageModel
import torch

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LEN,
    load_in_4bit=True,
)
print("loaded", MODEL_NAME)

## Litmus: can the *base* model already do it?

Run the base model on the held-out real questions and score it. We expect low spec-pass / alignment — the justification to fine-tune.

In [ ]:
import json
from src.prompts import SYSTEM_PROMPT, build_user, parse_distractors

gold = [json.loads(l) for l in open("data/processed/eval_heldout.jsonl")]

def generate_json(model, tokenizer, system, user, max_new_tokens=512):
    msgs = [{"role": "system", "content": system}, {"role": "user", "content": user}]
    ids = tokenizer.apply_chat_template(
        msgs, tokenize=True, add_generation_prompt=True,
        enable_thinking=False, return_tensors="pt",
    ).to(model.device)
    out = model.generate(input_ids=ids, max_new_tokens=max_new_tokens, do_sample=False)
    return tokenizer.decode(out[0][ids.shape[1]:], skip_special_tokens=True)

def run_predictions(model, tokenizer, gold, out_path):
    FastLanguageModel.for_inference(model)
    rows = []
    for i, r in enumerate(gold):
        txt = generate_json(model, tokenizer, SYSTEM_PROMPT, build_user(r["question"], r["correct"], r["topic"]))
        rows.append({"id": r["id"], "distractors": parse_distractors(txt)})
        if (i + 1) % 20 == 0:
            print(f"  {i + 1}/{len(gold)}")
    with open(out_path, "w") as f:
        for x in rows:
            f.write(json.dumps(x) + "\n")
    return rows

run_predictions(model, tokenizer, gold, "predictions_base.jsonl")
print("\n===== BASE model =====")
!python -m src.eval predictions_base.jsonl

## Fine-tune with QLoRA

LoRA on all linear layers, assistant-only loss, on the v1 dataset (1,341 examples).

In [ ]:
from unsloth.chat_templates import train_on_responses_only
from datasets import load_dataset
from trl import SFTTrainer, SFTConfig

model = FastLanguageModel.get_peft_model(
    model, r=32, lora_alpha=32, lora_dropout=0.0,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    use_gradient_checkpointing="unsloth", random_state=42,
)

def to_text(ex):
    msgs = [
        {"role": "system", "content": ex["system"]},
        {"role": "user", "content": ex["user"]},
        {"role": "assistant", "content": ex["assistant"]},
    ]
    return tokenizer.apply_chat_template(msgs, tokenize=False, enable_thinking=False)

ds = load_dataset("json", data_files="data/processed/train_v1.jsonl", split="train")
ds = ds.map(lambda ex: {"text": to_text(ex)})

trainer = SFTTrainer(
    model=model, tokenizer=tokenizer, train_dataset=ds,
    args=SFTConfig(
        dataset_text_field="text", max_seq_length=MAX_SEQ_LEN,
        per_device_train_batch_size=2, gradient_accumulation_steps=4,
        warmup_steps=5, num_train_epochs=EPOCHS, learning_rate=2e-4,
        logging_steps=10, optim="adamw_8bit", weight_decay=0.01,
        lr_scheduler_type="linear", seed=42, output_dir="outputs", report_to="none",
    ),
)
# Train only on the assistant response (Qwen3 chat markers).
trainer = train_on_responses_only(
    trainer,
    instruction_part="<|im_start|>user\n",
    response_part="<|im_start|>assistant\n",
)
trainer.train()

## Evaluate the tuned model (base vs tuned)

In [ ]:
run_predictions(model, tokenizer, gold, "predictions_tuned.jsonl")
print("\n===== TUNED model =====")
!python -m src.eval predictions_tuned.jsonl
print("\n===== BASE model (for comparison) =====")
!python -m src.eval predictions_base.jsonl

## Save the adapter (and optionally push to Hugging Face)

In [ ]:
model.save_pretrained("qwen3-1.7b-distractor-lora")
tokenizer.save_pretrained("qwen3-1.7b-distractor-lora")
print("saved adapter -> qwen3-1.7b-distractor-lora/")

# Optional: push to Hugging Face (set your token first)
# from huggingface_hub import login; login(token="hf_...")
# model.push_to_hub("jsonjj/qwen3-1.7b-distractor-lora")
# tokenizer.push_to_hub("jsonjj/qwen3-1.7b-distractor-lora")

### Notes
- Expect **tuned spec-pass and alignment >> base** — that gap is the core result.
- Judge-based consistency (Claude Sonnet 5 via TrueFoundry) needs a key: `!python -m src.eval predictions_tuned.jsonl --judge` (costs API).
- If an Unsloth/TRL API differs on Colab, mirror the current Unsloth Qwen3 notebook at https://unsloth.ai/docs.